In [ ]:
import matplotlib.cm as cm
from matplotlib.colors import hsv_to_rgb, Normalize
import matplotlib.pyplot as plt
from moviepy.editor import VideoClip
from moviepy.editor import *
import os
import math
import numpy as np
from tqdm.notebook import tqdm
import imageio

In [ ]:
def disparity_map_to_image(disparity, mapper=None):
    """
    Convert a disparity map to an RGB image.
    Source: https://github.com/uzh-rpg/DSEC/blob/main/scripts/dataset/visualization.py

    Args:
        disparity (np.ndarray): Disparity map with shape (1, height, width).

    Returns:
        np.ndarray: RGB image of the disparity map.
    """

    # check shape
    assert disparity.ndim == 3 and disparity.shape[
        0] == 1, "Disparity must have shape (1, height, width)."

    # disparity magnitude for nonzero pixels
    disparity = disparity.squeeze(0)  # remove channel
    disp_pixels = np.argwhere(disparity > 0)
    y, x = disp_pixels[:, 0], disp_pixels[:, 1]
    disp_valid = disparity[y, x]
    min_disp = disp_valid.min() if len(disp_valid) > 0 else 0
    max_disp = disp_valid.max() if len(disp_valid) > 0 else 0

    # disparity colormap (in reverse if depth map)
    norm = Normalize(vmin=min_disp, vmax=max_disp, clip=True)
    if mapper is None:
        mapper = cm.ScalarMappable(norm=norm, cmap="inferno")
    disp_color = mapper.to_rgba(disp_valid)[..., :3]
    image = np.zeros((*disparity.shape, 3))

    # to rgb ints
    image[y, x] = disp_color
    image = (image * 255).astype(np.uint8)

    return image, mapper


In [ ]:
class DSECDispSeq:
    fudge_number = 5
    def __init__(self, seq):
        self.seq = seq
        self.images_directory = os.path.join(f"/data/dsec-e/test/{seq}/images")
        self.ours_disp_directory = os.path.join(f"/home/yilun/dsec_disp_ours_full/{seq}")
        self.choo_et_al_disp_directory = os.path.join(f"/home/yilun/TemporalEventStereo/exp_debug_3/test/{seq}")
        self.index_files()

    def index_files(self):
        self.images = sorted(os.listdir(self.images_directory))
        self.ours_disps = sorted(os.listdir(self.ours_disp_directory))
        self.choo_et_al_disps = sorted(os.listdir(self.choo_et_al_disp_directory))
        min_index = max([math.ceil(float(x[0].rstrip('.png'))) for x in [self.images, self.ours_disps, self.choo_et_al_disps]]) + self.fudge_number
        max_index = min([math.floor(float(x[-1].rstrip('.png'))) for x in [self.images, self.ours_disps, self.choo_et_al_disps]])
        self.ours_begin_index = np.abs(np.array([float(x.rstrip('.png')) for x in self.ours_disps]) - min_index).argmin()
        self.ours_end_index = np.abs(np.array([float(x.rstrip('.png')) for x in self.ours_disps]) - max_index).argmin()

    def __len__(self):
        return self.ours_end_index - self.ours_begin_index + 1

    def __getitem__(self, index):
        test_index = math.ceil(float(self.ours_disps[index].rstrip('.png')))
        image = self.images[test_index]
        ours_disp = self.ours_disps[index]
        choo_et_al_disp = self.choo_et_al_disps[test_index - self.fudge_number]
        return image, ours_disp, choo_et_al_disp
    
    def get_image(self, index):
        test_index = math.ceil(float(self.ours_disps[index].rstrip('.png')))
        image = os.path.join(self.images_directory, self.images[test_index])
        return imageio.imread(image)
    
    def get_ours_disp(self, index):
        ours_disp = os.path.join(self.ours_disp_directory, self.ours_disps[index])
        disparity = imageio.imread(ours_disp).astype(np.float32) / 256
        disparity_img, mapper = disparity_map_to_image(np.expand_dims(disparity, 0), mapper=self.mapper)
        return disparity_img
    
    def get_choo_et_al_disp(self, index):
        test_index = math.ceil(float(self.ours_disps[index].rstrip('.png'))) - self.fudge_number
        choo_et_al_disp = os.path.join(self.choo_et_al_disp_directory, self.choo_et_al_disps[test_index])
        disparity = imageio.imread(choo_et_al_disp).astype(np.float32) / 256
        disparity_img, self.mapper = disparity_map_to_image(np.expand_dims(disparity, 0))
        return disparity_img



In [ ]:
for seq in os.listdir("/home/yilun/dsec_disp_ours_full"):
    x = DSECDispSeq(seq)
    fps = 30
    vid = clips_array([[
        VideoClip(lambda t: x.get_image(int(t*fps)), duration=x.__len__()/fps).resize(height=360),
        VideoClip(lambda t: x.get_choo_et_al_disp(int(t*fps)), duration=x.__len__()/fps).resize(height=360),
        VideoClip(lambda t: x.get_ours_disp(int(t*fps)), duration=x.__len__()/fps).resize(height=360),
    ]])
    vid.write_videofile(f"{seq}.mp4", fps=fps)